# Simulation Summary

- Interactive: open in Jupyter and **Run All**.
- Batch: `run_*.sh` auto-executes this notebook and saves output to `logs/summary_<TS>.txt`.
- Set `CASE_FILTER` in the setup cell to filter to a single case.
- `SUMMARY_OUTPUT` env var: captured output is written to that path.

Tables in order: Feasibility · Failure reasons · Totals · Position std · Velocity std · Position std error vs nominal · Velocity std error vs nominal · Overall nominal baseline.


In [ ]:
import matplotlib
matplotlib.use('Agg') if __import__('os').environ.get('SUMMARY_OUTPUT') else None
import matplotlib.pyplot as plt
import json, os, pickle
import numpy as np
from collections import Counter
from scipy import stats as sstats
from io import StringIO

# Edit for interactive single-case mode (e.g. 'case0_N5'); None = all cases
CASE_FILTER = None

SUMMARY_OUTPUT = os.environ.get('SUMMARY_OUTPUT')

METHODS = [('VO', 'vo'), ('FVO', 'fvo'), ('HO', 'ho'), ('MA', 'ma'), ('NOM', 'nominal')]
REASONS = ['ok', 'qp_infeasible', 'qp_iter_limit', 'qp_time_limit',
           'forward_invariance_fail', 'collision']
SHORT = {'ok': 'ok', 'qp_infeasible': 'infeas', 'qp_iter_limit': 'iter',
         'qp_time_limit': 'time', 'forward_invariance_fail': 'fi_fail',
         'collision': 'collis', 'qp_nan': 'nan'}

with open('cases.json') as f:
    cases = json.load(f)
if CASE_FILTER:
    cases = [c for c in cases if c['case_id'] == CASE_FILTER]
    if not cases:
        raise ValueError(f'CASE_FILTER={CASE_FILTER!r} not in cases.json')

case_ids = [c['case_id'] for c in cases]
# --- Plot styling constants (for merged plot cells) ---
CBF_METHODS = [(tag, m) for tag, m in METHODS if m != 'nominal']
COLOR = {'VO':'#4C72B0', 'FVO':'#DD8452', 'HO':'#55A467', 'MA':'#C44E52', 'NOM':'#8172B2'}
REASON_COLOR = {'ok':'#55A467', 'collision':'#C44E52', 'forward_invariance_fail':'#DD8452',
                'qp_infeasible':'#8172B2', 'qp_iter_limit':'#CCB974', 'qp_time_limit':'#64B5CD'}
REASON_SHORT = {'ok':'ok', 'collision':'collis', 'forward_invariance_fail':'fi_fail',
                'qp_infeasible':'infeas', 'qp_iter_limit':'iter', 'qp_time_limit':'time'}


_buf = StringIO()

def emit(msg=''):
    print(msg)
    print(msg, file=_buf)

emit(f"cases: {[c['case_id'] for c in cases]}")

In [ ]:
def _load_pkl(case_id, method):
    p = f'result/{case_id}/{method}/{method}_simulation_data.pkl'
    if not os.path.exists(p):
        return None
    with open(p, 'rb') as f:
        return pickle.load(f)

results = {(c['case_id'], m): _load_pkl(c['case_id'], m)
           for c in cases for _, m in METHODS}
missing = [k for k, v in results.items() if v is None]
if missing:
    emit(f'  [warn] missing pkl: {missing}')

In [ ]:
emit()
emit('  Feasibility (ok/total)')
emit(f"  {'case_id':<14} | " + " | ".join(f"{tag:>7}" for tag, _ in METHODS))
emit("  " + "-" * (16 + len(METHODS) * 11))
for case in cases:
    cid = case['case_id']
    row = [f"  {cid:<14}"]
    for tag, m in METHODS:
        d = results.get((cid, m))
        if d is None:
            row.append(f"{'N/A':>7}"); continue
        fl = d.get(f'{m}_feasibility_list', [])
        if fl:
            ok = sum(1 for x in fl if x)
            row.append(f"{ok:>3}/{len(fl):<3}")
        else:
            row.append(f"{'N/A':>7}")
    emit(" | ".join(row))

In [ ]:
emit()
emit('  Failure reason breakdown (count per method per case)')
_rhdr = (f"  {'case':<14} {'mthd':<4} "
         + " ".join(f"{SHORT[r]:>7}" for r in REASONS) + "  other")
emit(_rhdr)
emit("  " + "-" * (len(_rhdr) - 2))

totals = {}
for case in cases:
    cid = case['case_id']
    for tag, m in METHODS:
        d = results.get((cid, m))
        if d is None:
            emit(f"  {cid:<14} {tag:<4} (no pkl)"); continue
        rl = d.get(f'{m}_failure_reason_list', [])
        c = Counter(rl)
        cells_str = " ".join(f"{c.get(r, 0):>7}" for r in REASONS)
        other = sum(v for k, v in c.items() if k not in REASONS and k is not None)
        emit(f"  {cid:<14} {tag:<4} {cells_str} {other:>6}")
        for r in REASONS:
            totals[(tag, r)] = totals.get((tag, r), 0) + c.get(r, 0)
        totals[(tag, 'other')] = totals.get((tag, 'other'), 0) + other

In [ ]:
emit()
emit('  Totals per method across all cases')
emit(_rhdr)
emit("  " + "-" * (len(_rhdr) - 2))
for tag, _ in METHODS:
    cells_str = " ".join(f"{totals.get((tag, r), 0):>7}" for r in REASONS)
    other = totals.get((tag, 'other'), 0)
    emit(f"  {'':<14} {tag:<4} {cells_str} {other:>6}")

In [ ]:
_STD_WINDOW_STEPS = 200   # last 10 s @ DT=0.05s

def _window_std_per_run(history, axis_idx, n_steps=_STD_WINDOW_STEPS):
    """Avg std over the last n_steps timesteps for one run's history.
    axis_idx=0 → position slice [0:2], axis_idx=2 → velocity slice [2:4].
    history shape: (T+1, 5, N)."""
    if history is None:
        return None
    arr = np.asarray(history, dtype=float)
    n_agents = arr.shape[-1]
    tail = arr[-n_steps:, axis_idx:axis_idx+2, :]          # (n_steps, 2, N)
    mean = tail.mean(axis=-1, keepdims=True)               # (n_steps, 2, 1)
    # RMS deviation per timestep (matches calculate_std_dev in src notebooks)
    stds_per_t = np.sqrt(((tail - mean)**2).sum(axis=(-2, -1)) / n_agents)  # (n_steps,)
    return float(stds_per_t.mean())

def _mean_final_std(d, method, axis):
    """axis: 'position' | 'velocity'. Returns mean of last-10s-avg std over feasible runs, or None.
    Computed on-the-fly from `<method>_total_history` (robust to end-time oscillation)."""
    if d is None:
        return None
    fl = d.get(f'{method}_feasibility_list', [])
    hl = d.get(f'{method}_total_history', [])
    if not fl or not hl:
        return None
    axis_idx = 0 if axis == 'position' else 2
    vals = []
    for hist, ok in zip(hl, fl):
        if not ok:
            continue
        v = _window_std_per_run(hist, axis_idx)
        if v is not None and not np.isnan(v):
            vals.append(v)
    if not vals:
        return None
    return float(np.mean(vals))

def _fmt(v, w=8):
    return f"{'N/A':>{w}}" if v is None else f"{v:>{w}.3f}"

emit()
emit('  Position std (last-10s-avg, mean over feasible runs)')
emit(f"  {'case_id':<14} | " + " | ".join(f"{tag:>8}" for tag, _ in METHODS))
emit("  " + "-" * (16 + len(METHODS) * 11))
for case in cases:
    cid = case['case_id']
    row = [f"  {cid:<14}"]
    for tag, m in METHODS:
        row.append(_fmt(_mean_final_std(results.get((cid, m)), m, 'position')))
    emit(" | ".join(row))

In [ ]:
emit()
emit('  Velocity std (last-10s-avg, mean over feasible runs)')
emit(f"  {'case_id':<14} | " + " | ".join(f"{tag:>8}" for tag, _ in METHODS))
emit("  " + "-" * (16 + len(METHODS) * 11))
for case in cases:
    cid = case['case_id']
    row = [f"  {cid:<14}"]
    for tag, m in METHODS:
        row.append(_fmt(_mean_final_std(results.get((cid, m)), m, 'velocity')))
    emit(" | ".join(row))

In [ ]:
CBF_METHODS = [(tag, m) for tag, m in METHODS if m != 'nominal']

def _u_dev_stats(d, method, n_steps=_STD_WINDOW_STEPS):
    """Per feasible run:
         diff[t,i]   = |u_qp[t,i,0] - u_nom[t,i,0]|
         swarm_mean[t] = mean_i(diff[t,:])
         swarm_std[t]  = std_i(diff[t,:])
         run_mean = mean_t(swarm_mean over last n_steps)
         run_std  = mean_t(swarm_std  over last n_steps)
       Aggregate across feasible runs:
         return (mean_of_run_mean, mean_of_run_std, n_runs) or None.
    """
    if d is None:
        return None
    fl = d.get(f'{method}_feasibility_list', [])
    ch = d.get(f'{method}_total_control_history', [])
    nh = d.get(f'{method}_total_u_nominal_history', [])
    if not fl or not ch or not nh:
        return None
    run_means, run_stds = [], []
    for u_qp, u_nom, ok in zip(ch, nh, fl):
        if not ok or u_qp is None or u_nom is None:
            continue
        a = np.asarray(u_qp, dtype=float)
        b = np.asarray(u_nom, dtype=float)
        if a.shape != b.shape or a.ndim < 2:
            continue
        # (T, N, 1) or (T, N) -> collapse last singleton dim
        if a.ndim == 3 and a.shape[-1] == 1:
            a = a[..., 0]; b = b[..., 0]
        tail = min(n_steps, a.shape[0])
        if tail == 0:
            continue
        diff = np.abs(a[-tail:] - b[-tail:])               # (tail, N)
        swarm_mean_t = diff.mean(axis=1)                   # (tail,)
        swarm_std_t  = diff.std(axis=1, ddof=0)            # (tail,)
        run_means.append(float(swarm_mean_t.mean()))
        run_stds.append(float(swarm_std_t.mean()))
    if not run_means:
        return None
    return (float(np.mean(run_means)), float(np.mean(run_stds)), len(run_means))

def _fmt_dev(stats, w=15):
    if stats is None:
        return f"{'N/A':>{w}}"
    m, s, _ = stats
    return f"{m:>6.3f}±{s:<6.3f}".rjust(w)

emit()
emit('  Control deviation |u_qp - u_nom| per-agent (last-10s-avg  mean ± swarm-std, feasible runs)')
emit(f"  {'case_id':<14} | " + " | ".join(f"{tag:>15}" for tag, _ in CBF_METHODS))
emit("  " + "-" * (16 + len(CBF_METHODS) * 18))
for case in cases:
    cid = case['case_id']
    row = [f"  {cid:<14}"]
    for tag, m in CBF_METHODS:
        row.append(_fmt_dev(_u_dev_stats(results.get((cid, m)), m)))
    emit(" | ".join(row))


In [ ]:
CBF_METHODS = [(tag, m) for tag, m in METHODS if m != 'nominal']

def _timing_stats_ms(d, method, key_suffix):
    """Return (mean, p95, max) in ms, pooled over feasible runs. None if no data."""
    if d is None:
        return None
    fl = d.get(f'{method}_feasibility_list', [])
    tl = d.get(f'{method}_total_{key_suffix}', [])
    if not fl or not tl:
        return None
    flat = []
    for traj, ok in zip(tl, fl):
        if not ok or traj is None:
            continue
        arr = np.asarray(traj, dtype=float)
        arr = arr[~np.isnan(arr)]
        if arr.size:
            flat.append(arr)
    if not flat:
        return None
    a = np.concatenate(flat) * 1000.0  # seconds -> ms
    return (float(a.mean()), float(np.percentile(a, 95)), float(a.max()))

def _fmt_timing(stats):
    if stats is None:
        return f"{'N/A':>20}"
    m, p95, mx = stats
    return f"{m:>5.3f}/{p95:>5.3f}/{mx:>6.2f}"

# ----- Wall-clock wrapper timing -----
emit()
emit('  QP solve time (pure DAQP solver, ms)     format = mean / p95 / max')
emit(f"  {'case_id':<14} | " + " | ".join(f"{tag:>20}" for tag, _ in CBF_METHODS))
emit("  " + "-" * (16 + len(CBF_METHODS) * 23))
for case in cases:
    cid = case['case_id']
    row = [f"  {cid:<14}"]
    for tag, m in CBF_METHODS:
        row.append(_fmt_timing(_timing_stats_ms(results.get((cid, m)), m, 'daqp_solve_time_list')))
    emit(" | ".join(row))


In [ ]:
all_pos, all_vel, n_case = [], [], 0
for case in cases:
    d = results.get((case['case_id'], 'nominal'))
    if d is None:
        continue
    fl = d.get('nominal_feasibility_list', [])
    hl = d.get('nominal_total_history', [])
    if not fl or not hl:
        continue
    for hist, ok in zip(hl, fl):
        if not ok: continue
        vp = _window_std_per_run(hist, 0)
        vv = _window_std_per_run(hist, 2)
        if vp is not None and not np.isnan(vp): all_pos.append(vp)
        if vv is not None and not np.isnan(vv): all_vel.append(vv)
    n_case += 1

emit()
emit('  Overall nominal baseline (all feasible runs across all cases)')
if all_pos:
    emit(f"  Position std:  mean={np.mean(all_pos):.4f}  std={np.std(all_pos):.4f}  "
         f"min={np.min(all_pos):.4f}  max={np.max(all_pos):.4f}  (n={len(all_pos)} runs, {n_case} cases)")
else:
    emit('  Position std: (no nominal data)')
if all_vel:
    emit(f"  Velocity std:  mean={np.mean(all_vel):.4f}  std={np.std(all_vel):.4f}  "
         f"min={np.min(all_vel):.4f}  max={np.max(all_vel):.4f}  (n={len(all_vel)} runs)")
else:
    emit('  Velocity std: (no nominal data)')
emit()

In [ ]:
if SUMMARY_OUTPUT:
    os.makedirs(os.path.dirname(SUMMARY_OUTPUT) or '.', exist_ok=True)
    with open(SUMMARY_OUTPUT, 'w') as f:
        f.write(_buf.getvalue())
    print(f'  [saved to {SUMMARY_OUTPUT}]')

---
## Plots (interactive)

The cells below produce matplotlib figures. They are skipped in batch execution
(when `run_*.sh` invokes this notebook to save `logs/summary_*.txt`) since the
Agg backend produces figures without display. Run interactively in Jupyter to view charts.


## 1. Feasibility — bar chart (grouped by case)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(case_ids))
width = 0.15
for i, (tag, m) in enumerate(METHODS):
    rates = []
    for cid in case_ids:
        d = results.get((cid, m))
        fl = d.get(f'{m}_feasibility_list', []) if d else []
        rates.append(100 * sum(fl) / len(fl) if fl else np.nan)
    ax.bar(x + (i - 2) * width, rates, width, label=tag, color=COLOR[tag])
ax.set_xticks(x)
ax.set_xticklabels(case_ids)
ax.set_ylabel('Feasibility (%)'); ax.set_ylim(0, 105)
ax.set_title('Feasibility Rate per Method per Case')
ax.legend(loc='lower left', ncol=5, frameon=False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## 2. Failure reason breakdown — stacked bars

In [ ]:
fig, axes = plt.subplots(1, len(CBF_METHODS)+1, figsize=(16, 4.5), sharey=True)
for ax, (tag, m) in zip(axes, METHODS):
    bottoms = np.zeros(len(case_ids))
    for r in REASONS:
        counts = []
        for cid in case_ids:
            d = results.get((cid, m))
            if d is None:
                counts.append(0); continue
            rl = d.get(f'{m}_failure_reason_list', [])
            counts.append(Counter(rl).get(r, 0))
        ax.bar(range(len(case_ids)), counts, bottom=bottoms,
               label=REASON_SHORT[r], color=REASON_COLOR[r], edgecolor='white', linewidth=0.5)
        bottoms += counts
    ax.set_title(tag); ax.set_xticks(range(len(case_ids)))
    ax.set_xticklabels(case_ids, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('count' if tag == 'VO' else '')
    ax.grid(axis='y', alpha=0.3)
axes[0].legend(loc='upper right', fontsize=8, ncol=2, frameon=False)
plt.suptitle('Failure Reason Breakdown (per method per case)', y=1.02)
plt.tight_layout(); plt.show()


## 3. Position std — mean ± 95% CI (feasible runs only)

In [ ]:
_STD_WINDOW_STEPS = 200   # last 10 s @ DT=0.05s

def _window_std_per_run(history, axis_idx, n_steps=_STD_WINDOW_STEPS):
    if history is None: return None
    arr = np.asarray(history, dtype=float)
    n = arr.shape[-1]
    tail = arr[-n_steps:, axis_idx:axis_idx+2, :]
    mean = tail.mean(axis=-1, keepdims=True)
    stds = np.sqrt(((tail - mean)**2).sum(axis=(-2, -1)) / n)
    return float(stds.mean())

def collect_feasible(cid, m, key):
    """Return per-run stat for metric `key`, filtered to feasible runs.
    For `final_{axis}_std_dev_list`, compute last-10s-avg from history instead of single-point."""
    d = results.get((cid, m))
    if d is None: return []
    fl = d.get(f'{m}_feasibility_list', [])
    if key in ('final_position_std_dev_list', 'final_velocity_std_dev_list'):
        hl = d.get(f'{m}_total_history', [])
        axis_idx = 0 if 'position' in key else 2
        vals = []
        for hist, ok in zip(hl, fl):
            if not ok: continue
            v = _window_std_per_run(hist, axis_idx)
            if v is not None and not np.isnan(v):
                vals.append(v)
        return vals
    # Fallback for non-std metrics
    sl = d.get(f'{m}_{key}', [])
    return [s for s, ok in zip(sl, fl) if ok]

def mean_ci(vals, conf=0.95):
    if len(vals) < 2: return (np.mean(vals) if vals else np.nan, 0.0)
    m = float(np.mean(vals))
    sem = float(np.std(vals, ddof=1) / np.sqrt(len(vals)))
    ci = sem * sstats.t.ppf((1 + conf)/2, len(vals) - 1)
    return m, ci

def plot_metric(metric_key, title, ax):
    x = np.arange(len(case_ids))
    width = 0.15
    for i, (tag, m) in enumerate(METHODS):
        means, errs = [], []
        for cid in case_ids:
            vals = collect_feasible(cid, m, metric_key)
            mu, ci = mean_ci(vals)
            means.append(mu); errs.append(ci)
        ax.bar(x + (i - 2) * width, means, width, yerr=errs, capsize=3,
               label=tag, color=COLOR[tag], edgecolor='black', linewidth=0.3)
    ax.set_xticks(x); ax.set_xticklabels(case_ids)
    ax.set_title(title); ax.legend(ncol=5, frameon=False, fontsize=9)
    ax.grid(axis='y', alpha=0.3)

fig, ax = plt.subplots(figsize=(11, 5))
plot_metric('final_position_std_dev_list', 'Final Position Std — mean ± 95% CI', ax)
ax.set_ylabel('Position std (feasible runs)')
plt.tight_layout(); plt.show()


## 4. Velocity std — mean ± 95% CI

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_metric('final_velocity_std_dev_list', 'Final Velocity Std — mean ± 95% CI', ax)
ax.set_ylabel('Velocity std (feasible runs)')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(case_ids))
width = 0.8 / max(1, len(CBF_METHODS))
colors = plt.cm.tab10(np.linspace(0, 1, len(CBF_METHODS)))
for i, (tag, m) in enumerate(CBF_METHODS):
    means, stds = [], []
    for cid in case_ids:
        stats = _u_dev_stats(results.get((cid, m)), m)
        if stats is None:
            means.append(np.nan); stds.append(0.0)
        else:
            means.append(stats[0]); stds.append(stats[1])
    offs = (i - (len(CBF_METHODS) - 1) / 2.0) * width
    ax.bar(x + offs, means, width * 0.95, yerr=stds, capsize=3,
           label=tag, color=colors[i])
ax.set_xticks(x)
ax.set_xticklabels(case_ids, rotation=15, ha='right')
ax.set_ylabel('|u_qp - u_nom|  (last-10s-avg swarm-mean, error=swarm-std)')
ax.set_title('Control deviation from nominal (per-agent mean ± across-agent std)')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
def _per_step_errors(d, method):
    """(T, R) array: rows = timesteps, cols = feasible runs.
       Each cell = mean over agents of |u_qp - u_nom| at that (t, run)."""
    if d is None:
        return None
    fl = d.get(f'{method}_feasibility_list', [])
    ch = d.get(f'{method}_total_control_history', [])
    nh = d.get(f'{method}_total_u_nominal_history', [])
    if not fl or not ch or not nh:
        return None
    traces = []
    for u_qp, u_nom, ok in zip(ch, nh, fl):
        if not ok or u_qp is None or u_nom is None:
            continue
        a = np.asarray(u_qp, dtype=float)
        b = np.asarray(u_nom, dtype=float)
        if a.shape != b.shape:
            continue
        if a.ndim == 3 and a.shape[-1] == 1:
            a = a[..., 0]; b = b[..., 0]
        traces.append(np.abs(a - b).mean(axis=1))   # (T,)
    if not traces:
        return None
    T_min = min(len(t) for t in traces)
    arr = np.stack([t[:T_min] for t in traces])     # (R, T)
    return arr.T                                     # (T, R)

_DT = 0.05
n_cases = len(case_ids)
n_cols = 2 if n_cases > 1 else 1
n_rows = int(np.ceil(n_cases / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4 * n_rows), squeeze=False)
axes_flat = axes.flatten()
colors = plt.cm.tab10(np.linspace(0, 1, len(CBF_METHODS)))

for ax, cid in zip(axes_flat, case_ids):
    for (tag, m), color in zip(CBF_METHODS, colors):
        errs = _per_step_errors(results.get((cid, m)), m)
        if errs is None or errs.size == 0:
            continue
        T, R = errs.shape
        t_axis = np.arange(T) * _DT
        mu = errs.mean(axis=1)
        sd = errs.std(axis=1, ddof=0)
        ax.plot(t_axis, mu, label=f'{tag} (n={R})', color=color, linewidth=1.3)
        ax.fill_between(t_axis, mu - sd, mu + sd, color=color, alpha=0.18)
    ax.set_title(cid)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('|u_qp - u_nom|  (mean per agent, rad/s)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='upper right')

for ax in axes_flat[n_cases:]:
    ax.axis('off')

fig.suptitle('Per-step control deviation from nominal  (line = mean over feasible runs, band = ±std)')
plt.tight_layout()
plt.show()
